# 입찰메이트 RAG — Retrieval 파이프라인 GCP v1
**담당**: Retrieval 파트 (한의정)

**실행 전 필수**: 아래 셀 순서대로 실행. 커널 재시작 후에는 CELL 0부터 다시.

| 셀 | 내용 |
|---|---|
| CELL 0 | 환경변수 + sys.path 설정 |
| CELL 1 | Pre-Check (GPU/경로 확인) |
| CELL 2 | 데이터 로드 + 스키마 정규화 |
| CELL 3 | 임베딩 모델 로드 (KURE-v1) |
| CELL 4 | ChromaDB 구축 / 로드 |
| CELL 5 | BM25 인덱스 구축 / 로드 |
| CELL 6 | 메타데이터 파싱 유틸 |
| CELL 7 | BidMateRetriever 클래스 + 인스턴스 생성 |
| CELL 8 | 동작 테스트 |
| CELL 9 | 배치 Eval 평가 |

In [2]:
# import chromadb
# client = chromadb.PersistentClient(path='/home/euijeong/2Team_Project/chroma_db')
# client.delete_collection('bidmate_retrieval_v1')
# print('삭제 완료')

NotFoundError: Collection [bidmate_retrieval_v1] does not exist

In [3]:
# import subprocess
# result = subprocess.run(
#     ['pip', 'install', '-t', '/home/euijeong/2Team_Project/hej/my_packages',
#      '--no-deps',
#      'pandas', 'chromadb', 'sentence-transformers', 'kiwipiepy', 'rank_bm25', 'tqdm'],
#     capture_output=True, text=True
# )
# print(result.stdout[-500:])
# print(result.stderr[-300:])

━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 4/6 [pandas]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [chromadb]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [chromadb]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [chromadb]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [chromadb]


ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device




In [3]:
# ======================================================================
# [CELL 0] 환경변수 + sys.path 설정 — 커널 재시작 후 반드시 먼저 실행
# ======================================================================
import os
os.environ['BIDMATE_ENV'] = 'gcp'
print(f'✅ BIDMATE_ENV = {os.environ["BIDMATE_ENV"]}')
print(f'✅ 실행 계정   = {os.getenv("USER")}')

✅ BIDMATE_ENV = gcp
✅ 실행 계정   = euijeong


In [4]:
# ======================================================================
# [CELL 1] Pre-Check — GPU / 경로 전수 확인
# ======================================================================
import re
import ast
import math
import json
import pickle
import logging
import difflib
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from kiwipiepy import Kiwi
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

print('=' * 60)
print('[Pre-Check] 환경 확인 시작')
print('=' * 60)

# ── 1. GPU / Device ──────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE   = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU : {gpu_name} | VRAM : {vram_gb:.1f} GB')
elif torch.backends.mps.is_available():
    DEVICE  = 'mps'
    vram_gb = 0
    print('✅ Device : Apple MPS (M2)')
else:
    DEVICE  = 'cpu'
    vram_gb = 0
    print('⚠️  Device : CPU')

# ── 2. BATCH_SIZE 자동 조정 ──────────────────────────────────────────
# ChromaDB upsert 시 OOM 방지용
if DEVICE == 'cuda':
    BATCH_SIZE = 64 if vram_gb >= 20 else 32
elif DEVICE == 'mps':
    BATCH_SIZE = 32
else:
    BATCH_SIZE = 8
print(f'✅ BATCH_SIZE : {BATCH_SIZE}')

# ── 3. 환경 감지 ─────────────────────────────────────────────────────
ENV = os.environ.get('BIDMATE_ENV', 'local')
print(f'✅ ENV : {ENV.upper()}')

# ── 4. 경로 설정 ─────────────────────────────────────────────────────
if ENV == 'colab':
    PROJECT_ROOT = Path('/content/drive/MyDrive/rfp_rag_project')
    CHUNKS_PATH  = PROJECT_ROOT / 'data' / 'chunks'
    CHROMA_PATH  = PROJECT_ROOT / 'chroma_db'
    BM25_PATH    = PROJECT_ROOT / 'data' / 'bm25_index.pkl'
    EVAL_PATH    = PROJECT_ROOT / 'eval'
    RESULT_DIR = PROJECT_ROOT / 'eval_results'
elif ENV == 'gcp':
    PROJECT_ROOT = Path('/home/euijeong/2Team_Project/hej')
    CHUNKS_PATH  = PROJECT_ROOT / 'chunks'
    CHROMA_PATH  = Path('/home/euijeong/2Team_Project/chroma_db')
    BM25_PATH    = PROJECT_ROOT / 'bm25_index.pkl'
    EVAL_PATH    = PROJECT_ROOT / 'eval'
    RESULT_DIR   = PROJECT_ROOT / 'eval_results'
else:  # local Mac M2
    PROJECT_ROOT = Path.home() / 'Desktop' / 'AI엔지니어_8기' / '프로젝트' / '중급'
    CHUNKS_PATH  = PROJECT_ROOT / 'data' / 'chunks'
    CHROMA_PATH  = PROJECT_ROOT / 'chroma_db'
    BM25_PATH    = PROJECT_ROOT / 'data' / 'bm25_index.pkl'
    EVAL_PATH    = PROJECT_ROOT / 'eval'
    RESULT_DIR = PROJECT_ROOT / 'eval_results'

# ── 5. 경로 존재 확인 ────────────────────────────────────────────────
print(f'✅ PROJECT_ROOT : {PROJECT_ROOT}')
print(f'✅ CHUNKS_PATH  : {"존재" if CHUNKS_PATH.exists() else "없음 ⚠️"}')
print(f'✅ ChromaDB 폴더: {"있음" if CHROMA_PATH.exists() else "없음 ⚠️"}')
print(f'✅ BM25 Index   : {"기존 사용" if BM25_PATH.exists() else "새로 생성 필요"}')
print(f'✅ Eval 폴더    : {"존재" if EVAL_PATH.exists() else "없음 ⚠️"}')

BM25_EXIST = BM25_PATH.exists()

# ── 6. 로깅 설정 ─────────────────────────────────────────────────────
RESULT_DIR.mkdir(parents=True, exist_ok=True)
log_path = RESULT_DIR / 'retrieval_GCP.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.FileHandler(log_path), logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

print('=' * 60)
print('[Pre-Check] 완료')
print('=' * 60)

/opt/jhub-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Pre-Check] 환경 확인 시작
✅ GPU : NVIDIA L4 | VRAM : 23.7 GB
✅ BATCH_SIZE : 64
✅ ENV : GCP
✅ PROJECT_ROOT : /home/euijeong/2Team_Project/hej
✅ CHUNKS_PATH  : 존재
✅ ChromaDB 폴더: 있음
✅ BM25 Index   : 기존 사용
✅ Eval 폴더    : 존재
[Pre-Check] 완료


In [5]:
# ======================================================================
# [CELL 2] 데이터 로드 + 스키마 정규화
# ======================================================================
# 데이터 파트 JSON 키 이름이 Retrieval 파이프라인 표준과 다르다.
# normalize_chunk()로 로드 시점에 한 번만 변환 → 이후 코드 수정 불필요.
#
# [변환 규칙]
#   chunk_text           → text
#   organization_cleaned → agency
#   original_name        → source_file
#   year (int)           → year (str)
#   has_table/has_number → False 기본값 삽입
# ======================================================================
if not CHUNKS_PATH.exists():
    raise FileNotFoundError(f'청크 폴더 없음: {CHUNKS_PATH}')

chunk_files = sorted(CHUNKS_PATH.glob('*.json'))
if not chunk_files:
    raise FileNotFoundError(f'JSON 파일 없음: {CHUNKS_PATH}')


def normalize_chunk(c: dict) -> dict:
    """데이터 파트 JSON 스키마 → Retrieval 파이프라인 표준 키로 정규화."""
    meta = dict(c.get('metadata', {}))
    if 'agency' not in meta:
        meta['agency'] = meta.get('organization_cleaned',
                         meta.get('organization_raw', '미지정'))
    if 'source_file' not in meta:
        meta['source_file'] = meta.get('original_name', '')
    if 'year' in meta:
        meta['year'] = str(meta['year'])
    meta.setdefault('has_table',  False)
    meta.setdefault('has_number', False)
    return {
        'chunk_id': c.get('chunk_id', ''),
        'text'    : c.get('chunk_text', c.get('text', '')),
        'metadata': meta,
    }


ALL_CHUNKS = []
for f in chunk_files:
    with open(f, 'r', encoding='utf-8') as fp:
        data = json.load(fp)
        raw  = data if isinstance(data, list) else [data]
        ALL_CHUNKS.extend([normalize_chunk(c) for c in raw])

print(f'✅ 청크 로드 : {len(ALL_CHUNKS):,}개')

✅ 청크 로드 : 16,248개


In [6]:
# ======================================================================
# [CELL 3] 임베딩 모델 로드 (KURE-v1)
# ======================================================================
# 확정 스택: nlpai-lab/KURE-v1 — 한국어 RFP 도메인 특화.
# HuggingFace 캐시 있으면 캐시 로드, 없으면 다운로드.
# ======================================================================
EMBED_MODEL_ID = 'nlpai-lab/KURE-v1'
embed_model    = SentenceTransformer(EMBED_MODEL_ID, device=DEVICE)
print(f'✅ 임베딩 모델 로드 : {EMBED_MODEL_ID}')

2026-05-20 01:37:07,447 [INFO] HTTP Request: HEAD https://huggingface.co/nlpai-lab/KURE-v1/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 01:37:07,465 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/nlpai-lab/KURE-v1/d14c8a9423946e268a0c9952fecf3a7aabd73bd9/modules.json "HTTP/1.1 200 OK"
2026-05-20 01:37:07,514 [INFO] HTTP Request: HEAD https://huggingface.co/nlpai-lab/KURE-v1/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 01:37:07,516 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-20 01:37:07,533 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/nlpai-lab/KURE-v1/d14c8a9423946e268a0c9952fecf3a7aabd73bd9/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-20 01:37:07,535 [INFO] Loading SentenceTransformer model from nlpai-lab/KURE-v1.
2026-05-20 01:37:07

✅ 임베딩 모델 로드 : nlpai-lab/KURE-v1


In [7]:
# ======================================================================
# [CELL 4] ChromaDB 구축 / 로드
# ======================================================================
# 데이터 파트 컬렉션(kh_fixed_1200_200_v2)은 ChromaDB default EF인
# all-MiniLM-L6-v2(영어 특화)로 인덱싱 → 한국어 RFP 검색에 부적합.
# Retrieval 파트 전용 컬렉션을 별도로 만들어 KURE-v1으로 직접 인덱싱.
#
# count > 0이면 기존 컬렉션 재사용, count = 0이면 전체 인덱싱.
# ======================================================================
COLLECTION_NAME = 'bidmate_retrieval_v1'  # Retrieval 파트 전용

chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = chroma_client.get_or_create_collection(
    name     = COLLECTION_NAME,
    metadata = {'hnsw:space': 'cosine'},
)

doc_count = collection.count()
print(f'✅ ChromaDB 컬렉션 : {COLLECTION_NAME} | 문서 수 : {doc_count:,}')

if doc_count == 0:
    # chunk_id 중복 제거
    seen_ids, deduped_chunks = set(), []
    for c in ALL_CHUNKS:
        if c['chunk_id'] not in seen_ids:
            seen_ids.add(c['chunk_id'])
            deduped_chunks.append(c)

    n_dup = len(ALL_CHUNKS) - len(deduped_chunks)
    if n_dup > 0:
        print(f'⚠️  중복 {n_dup}개 제거 : {len(ALL_CHUNKS):,} → {len(deduped_chunks):,}개')

    print(f'🔄 KURE-v1 임베딩 + ChromaDB 인덱싱 시작 | 총 {len(deduped_chunks):,}개')
    for i in range(0, len(deduped_chunks), BATCH_SIZE):
        batch      = deduped_chunks[i : i + BATCH_SIZE]
        texts      = [c['text'] for c in batch]
        embeddings = embed_model.encode(texts, normalize_embeddings=True).tolist()
        collection.upsert(
            ids        = [c['chunk_id'] for c in batch],
            documents  = texts,
            metadatas  = [c['metadata'] for c in batch],
            embeddings = embeddings,
        )
        if (i // BATCH_SIZE) % 10 == 0:
            print(f'   진행 : {min(i + BATCH_SIZE, len(deduped_chunks)):,} / {len(deduped_chunks):,}')
    print(f'✅ ChromaDB 인덱싱 완료 | 문서 수 : {collection.count():,}')
else:
    print('✅ 기존 컬렉션 재사용 (인덱싱 스킵)')

✅ ChromaDB 컬렉션 : bidmate_retrieval_v1 | 문서 수 : 0
🔄 KURE-v1 임베딩 + ChromaDB 인덱싱 시작 | 총 16,248개


Batches: 100%|██████████| 2/2 [00:04<00:00,  2.05s/it]


   진행 : 64 / 16,248


Batches: 100%|██████████| 2/2 [00:04<00:00,  2.03s/it]


   진행 : 704 / 16,248


...

   진행 : 15,424 / 16,248


Batches: 100%|██████████| 2/2 [00:05<00:00,  2.75s/it]


   진행 : 16,064 / 16,248


Batches: 100%|██████████| 2/2 [00:03<00:00,  1.70s/it]


✅ ChromaDB 인덱싱 완료 | 문서 수 : 16,248


In [8]:
# ======================================================================
# [CELL 5] BM25 인덱스 구축 / 로드
# ======================================================================
# Sparse 검색 축 담당. pkl로 직렬화해두고 재사용.
# Kiwi 형태소 분석으로 한국어 최적화 토크나이징.
# ======================================================================
kiwi = Kiwi()


def tokenize_ko(text: str) -> list:
    """Kiwi 기반 한국어 토크나이저. 명사/동사/형용사/영문/숫자만 추출."""
    if not isinstance(text, str) or not text.strip():
        return []
    keep_tags = {'NNG', 'NNP', 'NNB', 'NR', 'VV', 'VA', 'SL', 'SN'}
    return [
        t.form for t in kiwi.tokenize(text, normalize_coda=True)
        if t.tag in keep_tags and len(t.form) > 1
    ]


if BM25_EXIST:
    with open(BM25_PATH, 'rb') as f:
        bm25_data = pickle.load(f)
    bm25_index     = bm25_data['index']
    bm25_chunk_ids = bm25_data['chunk_ids']
    bm25_texts     = bm25_data['texts']
    print(f'✅ 기존 BM25 인덱스 로드 | 문서 수 : {len(bm25_chunk_ids):,}')
else:
    texts_for_bm25 = [c['text']     for c in ALL_CHUNKS]
    bm25_chunk_ids = [c['chunk_id'] for c in ALL_CHUNKS]

    print(f'🔄 BM25 토크나이징 시작 | 총 {len(texts_for_bm25):,}개')
    tokenized_corpus = [tokenize_ko(t) for t in texts_for_bm25]

    if not any(tokenized_corpus):
        raise ValueError('BM25 토크나이징 결과 전부 비어있음. 청크 텍스트 확인 필요.')

    bm25_index = BM25Okapi(tokenized_corpus)
    bm25_texts = texts_for_bm25

    BM25_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(BM25_PATH, 'wb') as f:
        pickle.dump({'index': bm25_index, 'chunk_ids': bm25_chunk_ids, 'texts': bm25_texts}, f)
    print('✅ BM25 인덱스 구축 및 저장 완료')

✅ 기존 BM25 인덱스 로드 | 문서 수 : 35,366


In [9]:
# ======================================================================
# [CELL 6] 메타데이터 파싱 유틸
# ======================================================================
# 질문에서 agency / year를 자동 추출해 Hard Filter로 사용.
# E타입(오타/약칭) 대응: SequenceMatcher 퍼지 매칭.
# ======================================================================
ALL_AGENCIES = list({
    c['metadata'].get('agency', '') for c in ALL_CHUNKS
    if c['metadata'].get('agency', '')
})


def extract_year(query: str):
    for pat in [r'(20\d{2})년?', r"'(\d{2})년?", r'(\d{2})년도']:
        m = re.search(pat, query)
        if m:
            y = m.group(1)
            return y if len(y) == 4 else f'20{y}'
    return None


def fuzzy_match_agency(query: str, threshold: float = 0.6):
    best_match, best_score = None, 0.0
    for agency in ALL_AGENCIES:
        score = difflib.SequenceMatcher(None, query, agency).ratio()
        for keyword in agency.split():
            if len(keyword) >= 2 and keyword in query:
                score = max(score, 0.75)
        if score > best_score:
            best_score = score
            best_match = agency
    return best_match if best_score >= threshold else None


def parse_metadata_filter(query: str) -> dict:
    filters = {}
    agency = fuzzy_match_agency(query)
    if agency: filters['agency'] = agency
    year = extract_year(query)
    if year: filters['year'] = year
    return filters


print(f'✅ 기관 목록 로드 : {len(ALL_AGENCIES)}개')
# 동작 확인
test_q = '한국가스공사 2024년 스마트 미터링 사업 예산은?'
print(f'테스트 필터 : {parse_metadata_filter(test_q)}')

✅ 기관 목록 로드 : 404개
테스트 필터 : {'agency': '한국가스공사', 'year': '2024'}


In [10]:
# ======================================================================
# [CELL 7] BidMateRetriever v5 + 인스턴스 생성
# ======================================================================
# 파이프라인 구조:
#   ① Hard Filter → ①-1 쿼리 분해 → ② Dense + ③ Sparse
#   → ④ RRF → ⑤ Soft Boost → ⑤-1 MMR → ⑥-1 Reranker → ⑦ Top-5
# ======================================================================

DENSE_K      = 15
SPARSE_K     = 15
RRF_K        = 60
TOP_K        = 5
MMR_LAMBDA   = 0.6
MMR_TOP_N    = 20
RERANK_TOP_N = 15


class BidMateRetriever:
    def __init__(self, collection, bm25_index, bm25_chunk_ids,
                 bm25_texts, embed_model, all_chunks: list, reranker=None):
        self.collection     = collection
        self.bm25_index     = bm25_index
        self.bm25_chunk_ids = bm25_chunk_ids
        self.bm25_texts     = bm25_texts
        self.embed_model    = embed_model
        self.chunk_meta_map = {c['chunk_id']: c['metadata'] for c in all_chunks}
        self.chunk_text_map = {c['chunk_id']: c['text']     for c in all_chunks}
        self._emb_cache: dict = {}
        self.reranker = reranker

    def _build_chroma_where(self, meta_filter: dict):
        if not meta_filter: return None
        conditions = []
        for key, val in meta_filter.items():
            if not val: continue
            if isinstance(val, dict): conditions.append({key: val})
            elif isinstance(val, list): conditions.append({key: {'$in': [str(v) for v in val]}})
            else: conditions.append({key: {'$eq': str(val)}})
        if not conditions: return None
        return conditions[0] if len(conditions) == 1 else {'$and': conditions}

    def _filter_bm25_ids(self, meta_filter: dict):
        if not meta_filter: return None
        allowed = set()
        for i, cid in enumerate(self.bm25_chunk_ids):
            meta = self.chunk_meta_map.get(cid, {})
            if 'agency' in meta_filter and meta.get('agency') != meta_filter['agency']: continue
            if 'year'   in meta_filter and meta.get('year')   != meta_filter['year']:   continue
            allowed.add(i)
        return list(allowed) if allowed else None

    def _decompose_query(self, query: str) -> list:
        """Agency 위치 앵커링 기반 쿼리 분해.
        복수 기관 B타입 질문을 기관별 서브쿼리로 분해.
        단일 기관이면 원본 쿼리 그대로 반환.
        """
        clean_query     = query.replace("'", '').replace('"', '')
        sorted_agencies = sorted(ALL_AGENCIES, key=len, reverse=True)
        found_agencies  = []
        for agency in sorted_agencies:
            agency_core = re.sub(r'^\(주\)|\(사\)|주식회사', '', agency).strip()
            if len(agency_core) >= 2 and agency_core in clean_query:
                if not any(agency_core in fa[1] for fa in found_agencies):
                    found_agencies.append((agency, agency_core))
        if len(found_agencies) < 2: return [query]
        found_agencies = found_agencies[:3]
        found_agencies.sort(key=lambda x: clean_query.find(x[1]))
        segments = []
        for i, (agency_full, agency_core) in enumerate(found_agencies):
            start_idx    = clean_query.find(agency_core) + len(agency_core)
            end_idx      = clean_query.find(found_agencies[i+1][1]) if i < len(found_agencies)-1 else len(clean_query)
            segment_text = clean_query[start_idx:end_idx]
            tokens = kiwi.tokenize(segment_text, normalize_coda=True)
            nouns  = [t.form for t in tokens if (t.tag.startswith('N') or t.tag in ['SL','SN']) and len(t.form) >= 2]
            segments.append({'agency': agency_full, 'local_nouns': nouns})
        global_nouns = segments[-1]['local_nouns']
        sub_queries  = []
        for seg in segments:
            local_nouns = seg['local_nouns']
            final_nouns = global_nouns if not local_nouns and global_nouns else local_nouns
            sub_queries.append(f"{seg['agency']} {' '.join(final_nouns)}".strip())
        return sub_queries

    def _dense_search(self, query: str, where) -> list:
        q_emb  = self.embed_model.encode([query], normalize_embeddings=True).tolist()
        kwargs = dict(query_embeddings=q_emb, n_results=DENSE_K,
                      include=['documents','metadatas','distances','embeddings'])
        if where: kwargs['where'] = where
        results = self.collection.query(**kwargs)
        emb_list = results.get('embeddings')
        if emb_list is not None and len(emb_list) > 0 and len(emb_list[0]) > 0:
            for cid, emb in zip(results['ids'][0], emb_list[0]):
                if cid not in self._emb_cache: self._emb_cache[cid] = emb
        return results['ids'][0]

    def _multi_retrieve(self, queries, where, allowed_indices, original_query=''):
        all_queries    = ([original_query] if original_query else []) + list(queries)
        dense_ids_all, sparse_ids_all = [], []
        seen_dense, seen_sparse = set(), set()
        for q in all_queries:
            for cid in self._dense_search(q, where):
                if cid not in seen_dense: dense_ids_all.append(cid); seen_dense.add(cid)
            for cid in self._sparse_search(q, allowed_indices):
                if cid not in seen_sparse: sparse_ids_all.append(cid); seen_sparse.add(cid)
        return dense_ids_all, sparse_ids_all

    def _sparse_search(self, query: str, allowed_indices) -> list:
        tokens = tokenize_ko(query)
        if not tokens: return []
        scores = self.bm25_index.get_scores(tokens)
        if allowed_indices is not None:
            mask = np.zeros(len(scores))
            mask[allowed_indices] = 1.0
            scores = scores * mask
        top_indices = np.argsort(scores)[::-1][:SPARSE_K]
        return [self.bm25_chunk_ids[i] for i in top_indices if scores[i] > 0]

    def _rrf_fusion(self, dense_ids, sparse_ids):
        rrf_scores = {}
        for rank, cid in enumerate(dense_ids,  start=1): rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0/(RRF_K+rank)
        for rank, cid in enumerate(sparse_ids, start=1): rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0/(RRF_K+rank)
        return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    def _soft_boost(self, ranked):
        """has_table x1.10 / has_number x1.05. 데이터 파트 v2는 해당 키 없어 사실상 x1.0."""
        boosted = []
        for cid, score in ranked:
            meta = self.chunk_meta_map.get(cid, {})
            mul  = 1.0
            if meta.get('has_table',  False): mul += 0.10
            if meta.get('has_number', False): mul += 0.05
            boosted.append((cid, score * mul))
        return sorted(boosted, key=lambda x: x[1], reverse=True)

    def _mmr_rerank(self, boosted, query):
        """MMR 재정렬. B타입 RRF Crowding 보정. λ=0.6 (v9 확정값)."""
        candidates = boosted[:MMR_TOP_N]
        if len(candidates) <= 1: return candidates
        valid = [(cid, score) for cid, score in candidates if cid in self._emb_cache]
        if not valid: return candidates
        cids   = [cid for cid, _ in valid]
        scores = [score for _, score in valid]
        vecs   = np.array([self._emb_cache[cid] for cid in cids], dtype=np.float32)
        q_vec  = np.array(self.embed_model.encode([query], normalize_embeddings=True)[0], dtype=np.float32)
        norms     = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9
        vecs_norm = vecs / norms
        q_norm    = q_vec / (np.linalg.norm(q_vec) + 1e-9)
        rel_scores = vecs_norm @ q_norm
        def minmax(arr):
            mn, mx = arr.min(), arr.max()
            return (arr - mn) / (mx - mn + 1e-9)
        relevance     = (minmax(rel_scores) + minmax(np.array(scores))) / 2.0
        selected_idx  = []
        remaining_idx = list(range(len(cids)))
        while remaining_idx:
            if not selected_idx:
                best = max(remaining_idx, key=lambda i: relevance[i])
            else:
                sel_vecs = vecs_norm[selected_idx]
                best, best_mmr = -1, -float('inf')
                for i in remaining_idx:
                    mmr_score = MMR_LAMBDA * relevance[i] - (1-MMR_LAMBDA) * float(np.max(vecs_norm[i] @ sel_vecs.T))
                    if mmr_score > best_mmr: best_mmr = mmr_score; best = i
            selected_idx.append(best); remaining_idx.remove(best)
        return [(cids[i], scores[i]) for i in selected_idx]

    def _rerank(self, boosted, query, rerank_top_n=RERANK_TOP_N):
        """Cross-Encoder(bge-reranker-v2-m3) 정밀 재정렬. reranker=None이면 스킵."""
        if self.reranker is None: return boosted
        candidates = boosted[:rerank_top_n]
        if not candidates: return boosted
        pairs    = [[query, self.chunk_text_map.get(cid, '')] for cid, _ in candidates]
        scores   = self.reranker.predict(pairs, show_progress_bar=False)
        reranked = sorted(zip([cid for cid, _ in candidates], scores), key=lambda x: x[1], reverse=True)
        reranked_ids = {cid for cid, _ in reranked}
        tail = [(cid, score) for cid, score in boosted[rerank_top_n:] if cid not in reranked_ids]
        return [(cid, float(score)) for cid, score in reranked] + tail

    def _build_context(self, top_chunks):
        lines = []
        for idx, (cid, score) in enumerate(top_chunks[:TOP_K], start=1):
            text   = self.chunk_text_map.get(cid, '')
            meta   = self.chunk_meta_map.get(cid, {})
            lines.append(f'[{idx}] {text}\n(출처: {meta.get("agency","미상")} {meta.get("year","")} | score: {score:.4f})')
        return '\n\n'.join(lines)

    def retrieve(self, query: str, meta_filter=None, verbose: bool = False) -> dict:
        if meta_filter is None: meta_filter = parse_metadata_filter(query)
        if verbose: print(f'[①] 메타 필터 : {meta_filter}')
        where           = self._build_chroma_where(meta_filter)
        allowed_indices = self._filter_bm25_ids(meta_filter)
        sub_queries     = self._decompose_query(query)
        if verbose:
            if len(sub_queries) > 1:
                print(f'[①-1] 쿼리 분해 : {len(sub_queries)}개')
                for i, sq in enumerate(sub_queries, 1): print(f'       서브쿼리 {i} : {sq}')
            else: print('[①-1] 단일 쿼리')
        if len(sub_queries) > 1:
            dense_ids, sparse_ids = self._multi_retrieve(sub_queries, where, allowed_indices, original_query=query)
        else:
            dense_ids  = self._dense_search(query, where)
            sparse_ids = self._sparse_search(query, allowed_indices)
        if verbose: print(f'[②] Dense {len(dense_ids)}개 / [③] Sparse {len(sparse_ids)}개')
        ranked  = self._rrf_fusion(dense_ids, sparse_ids)
        boosted = self._soft_boost(ranked)
        boosted = self._mmr_rerank(boosted, query=query)
        boosted = self._rerank(boosted, query=query)
        if verbose: print(f'[⑦] Top-{TOP_K} 선정 완료')
        top5    = boosted[:TOP_K]
        return {
            'context'    : self._build_context(top5),
            'top_chunks' : [{'rank': i+1, 'chunk_id': cid, 'boosted_score': score,
                             'text': self.chunk_text_map.get(cid,''), 'metadata': self.chunk_meta_map.get(cid,{})}
                            for i, (cid, score) in enumerate(top5)],
            'meta_filter': meta_filter,
            'dense_ids'  : dense_ids,
            'sparse_ids' : sparse_ids,
            'sub_queries': sub_queries,
        }


# ── Reranker + 인스턴스 생성 ─────────────────────────────────────────
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=DEVICE)
print('✅ Reranker 로드 : BAAI/bge-reranker-v2-m3')

retriever = BidMateRetriever(
    collection=collection, bm25_index=bm25_index,
    bm25_chunk_ids=bm25_chunk_ids, bm25_texts=bm25_texts,
    embed_model=embed_model, all_chunks=ALL_CHUNKS, reranker=reranker,
)
print('✅ BidMateRetriever v5 (GCP) 초기화 완료')
print('\n' + '='*60)
print('✅ 전체 파이프라인 초기화 완료')
print('   retriever.retrieve(query) 로 검색 가능')
print('='*60)

2026-05-20 01:58:16,405 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-v2-m3/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-05-20 01:58:16,407 [INFO] No modules.json found for BAAI/bge-reranker-v2-m3, initializing a new CrossEncoder model.
2026-05-20 01:58:16,461 [INFO] HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3 "HTTP/1.1 200 OK"
2026-05-20 01:58:16,508 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-v2-m3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 01:58:16,527 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-v2-m3/953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e/config.json "HTTP/1.1 200 OK"
2026-05-20 01:58:16,575 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-v2-m3/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-05-20 01:58:16,624 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-v2-m3/resol

✅ Reranker 로드 : BAAI/bge-reranker-v2-m3
✅ BidMateRetriever v5 (GCP) 초기화 완료

✅ 전체 파이프라인 초기화 완료
   retriever.retrieve(query) 로 검색 가능


In [11]:
# ======================================================================
# [CELL 8] 동작 테스트
# ======================================================================
test_query = '한국가스공사 차세대 ERP 시스템 구축 사업 예산은?'
result = retriever.retrieve(test_query, verbose=True)
print('\n[컨텍스트]')
print(result['context'][:500], '...')

[①] 메타 필터 : {'agency': '한국가스공사'}
[①-1] 단일 쿼리


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.18it/s]


[②] Dense 6개 / [③] Sparse 10개


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.84it/s]


[⑦] Top-5 선정 완료

[컨텍스트]
[1] 과 업 지 시 서
- 차세대 통합정보시스템(ERP) 구축 -

2024. 8.

목 차

Ⅰ. 개요

Ⅱ. 과업내용

Ⅲ. 일반사항

 1. 사 업 명 : 차세대 통합정보시스템(ERP) 구축

2. 사업목적
 ○ 기술지원 종료(‘27년)에 대비한 ERP 업그레이드
 - 제조사(SAP社)의 기술지원 종료 이후 세법 개정, 보안취약점 발생시 원활한 대응이 불가 등 기술지원 종료 대비한 선제적 대응으로 안정적 서비스 제공

 ○ 정보시스템 복잡도 해소 및 접근‧활용 간소화로 업무생산성 향상
 - ‘09년 도입 이후 종합적인 개선 없이 단편적 수정으로 복잡도 증가 및 대사·검증 등 수작업에 따른 비효율 해소
 * 원가결산, 급여정산 등 검증을 위한 수작업 및 대용량 데이터 처리시 속도지연 발생
 - 시스템 접근, 데이터 탐색‧활용 간소화로 사용자 편의 및 데이터 기반의 업무환경 마련
 * KOGAS 디지털 전환 전략 수립 설문결과(임직원 45%, 데이터 탐색불가로 불편 호소)

3. ...


In [12]:
# ======================================================================
# [CELL 9] 배치 Eval 평가 — Hit@5 · MRR · nDCG
# ======================================================================

def normalize_filename(fn: str) -> str:
    return Path(fn).stem.strip() if fn else ''

def _match_flags(gt_stems, retrieved_stems):
    """1:1 매칭 강제 — nDCG > 1 방지."""
    remaining = list(gt_stems)
    flags = []
    for r in retrieved_stems:
        matched = False
        for gt in remaining:
            if gt in r or r in gt:
                flags.append(1); remaining.remove(gt); matched = True; break
        if not matched: flags.append(0)
    return flags

def calc_hit(gt_stems, retrieved_stems):
    if not gt_stems: return None
    return 1 in _match_flags(gt_stems, retrieved_stems)

def calc_mrr(gt_stems, retrieved_stems):
    if not gt_stems: return None
    for rank, f in enumerate(_match_flags(gt_stems, retrieved_stems), start=1):
        if f: return 1.0 / rank
    return 0.0

def calc_ndcg(gt_stems, retrieved_stems):
    if not gt_stems: return None
    flags = _match_flags(gt_stems, retrieved_stems)
    dcg   = sum(f / math.log2(r+1) for r, f in enumerate(flags, start=1))
    idcg  = sum(1.0/math.log2(i+2) for i in range(min(len(gt_stems), len(flags))))
    return dcg / idcg if idcg > 0 else 0.0


# ── Eval 로드 ─────────────────────────────────────────────────────────
eval_files = sorted(EVAL_PATH.glob('*.csv'))
if not eval_files:
    raise FileNotFoundError(f'Eval CSV 없음: {EVAL_PATH}')
eval_df = pd.concat([pd.read_csv(f) for f in eval_files], ignore_index=True)
print(f'✅ Eval 데이터 로드 : {len(eval_df):,}개')

# ── 평가 루프 ─────────────────────────────────────────────────────────
results_log = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc='Evaluating'):
    query  = row['question']
    gt_raw = row.get('ground_truth_docs', '')
    try:    gt_docs = ast.literal_eval(gt_raw) if isinstance(gt_raw, str) else []
    except: gt_docs = [gt_raw] if isinstance(gt_raw, str) and gt_raw else []
    mf_raw = row.get('metadata_filter', {})
    try:    meta_filter = ast.literal_eval(mf_raw) if isinstance(mf_raw, str) else {}
    except: meta_filter = {}
    result = retriever.retrieve(query, meta_filter=meta_filter or None)
    retrieved_stems = [normalize_filename(c['metadata'].get('source_file','')) for c in result['top_chunks']]
    gt_stems        = [normalize_filename(g) for g in gt_docs]
    results_log.append({
        'id': row['id'], 'type': row['type'], 'difficulty': row['difficulty'],
        'hit': calc_hit(gt_stems, retrieved_stems),
        'mrr': calc_mrr(gt_stems, retrieved_stems),
        'ndcg': calc_ndcg(gt_stems, retrieved_stems),
        'retrieved': retrieved_stems, 'gt_docs': gt_stems,
        'sub_queries': result['sub_queries'],
    })

result_df   = pd.DataFrame(results_log)
eval_subset = result_df[result_df['hit'].notna()]
oh, om, on_ = eval_subset['hit'].mean(), eval_subset['mrr'].mean(), eval_subset['ndcg'].mean()

print('\n' + '='*55)
print(f'📊 전체  Hit@5 : {oh:.4f}  ({oh*100:.2f}%)')
print(f'📊 전체  MRR   : {om:.4f}')
print(f'📊 전체  nDCG  : {on_:.4f}')
print('='*55)
print('\n[타입별]')
print(f'{"타입":>4}  {"Hit@5":>7} {"MRR":>7} {"nDCG":>7}  n')
for t, g in eval_subset.groupby('type'):
    print(f'  {t:>2}  {g["hit"].mean():>7.4f} {g["mrr"].mean():>7.4f} {g["ndcg"].mean():>7.4f}  {len(g)}')
print('\n[난이도별]')
print(f'{"난이도":>4}  {"Hit@5":>7} {"MRR":>7} {"nDCG":>7}  n')
for d, g in eval_subset.groupby('difficulty'):
    print(f'  {d:>2}  {g["hit"].mean():>7.4f} {g["mrr"].mean():>7.4f} {g["ndcg"].mean():>7.4f}  {len(g)}')

# ── 버전 자동 증가 저장 ───────────────────────────────────────────────
run_time = datetime.now().strftime('%Y-%m-%d %H:%M')
existing = list(RESULT_DIR.glob('eval_results_retrieval_v*.csv'))
nums = [int(f.stem.split('_v')[-1]) for f in existing if f.stem.split('_v')[-1].isdigit()]
next_v = max(nums) + 1 if nums else 1

summary_rows = [{'id': f'[SUMMARY] v{next_v} | {run_time} | n={len(eval_subset)}',
                 'type':'ALL','difficulty':'ALL','hit':round(oh,4),'mrr':round(om,4),'ndcg':round(on_,4),'retrieved':'','gt_docs':'','sub_queries':''}]
for t, g in eval_subset.groupby('type'):
    summary_rows.append({'id':f'[TYPE_{t}]','type':t,'difficulty':'ALL',
                         'hit':round(g['hit'].mean(),4),'mrr':round(g['mrr'].mean(),4),'ndcg':round(g['ndcg'].mean(),4),
                         'retrieved':'','gt_docs':f'n={len(g)}','sub_queries':''})
for d, g in eval_subset.groupby('difficulty'):
    summary_rows.append({'id':f'[DIFF_{d}]','type':'ALL','difficulty':d,
                         'hit':round(g['hit'].mean(),4),'mrr':round(g['mrr'].mean(),4),'ndcg':round(g['ndcg'].mean(),4),
                         'retrieved':'','gt_docs':f'n={len(g)}','sub_queries':''})

output_df   = pd.concat([pd.DataFrame(summary_rows), result_df], ignore_index=True)
RESULT_SAVE = RESULT_DIR / f'eval_results_retrieval_v{next_v}.csv'
output_df.to_csv(RESULT_SAVE, index=False, encoding='utf-8-sig')
print(f'\n✅ 평가 결과 저장 : {RESULT_SAVE}')
print(f'   (상단 {len(summary_rows)}행 = 요약 / 이후 {len(result_df)}행 = 개별 결과)')

✅ Eval 데이터 로드 : 1,100개


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.25it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.57it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 53.72it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.47it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 55.12it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 55.98it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.31it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.08it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 54.26it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.22it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.50it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.67it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 54.74it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 55.29it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 49.05it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.96it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 46.01it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 47.1


📊 전체  Hit@5 : 0.8264  (82.64%)
📊 전체  MRR   : 0.7375
📊 전체  nDCG  : 0.7043

[타입별]
  타입    Hit@5     MRR    nDCG  n
   A   0.9146  0.8414  0.8601  328
   B   0.8650  0.7924  0.6138  311
   C   0.7203  0.6311  0.6544  143
   D   0.7963  0.6908  0.7174  162
   E   0.6923  0.5551  0.5891  156

[난이도별]
 난이도    Hit@5     MRR    nDCG  n
   상   0.7470  0.6424  0.6279  328
   중   0.8149  0.7305  0.6870  389
  최상   0.0000  0.0000  0.0000  7
   하   0.9229  0.8413  0.8019  376

✅ 평가 결과 저장 : /home/euijeong/2Team_Project/hej/eval_results_retrieval_v1.csv
   (상단 10행 = 요약 / 이후 1100행 = 개별 결과)


In [13]:
# ======================================================================
# [CELL 10] Generation 파트 전용 인터페이스
# ======================================================================
# Generation 파트는 이 함수만 호출하면 됨.
# C타입: history의 마지막 user 발화를 쿼리 앞에 붙여 합성 (Hit@5 +15.33%p).
# ======================================================================
def get_context(query: str, history=None, meta_filter=None) -> dict:
    effective_query = query
    if history:
        prev_user = [h['content'] for h in history if h['role'] == 'user']
        if prev_user: effective_query = f'{prev_user[-1]} {query}'
    result = retriever.retrieve(effective_query, meta_filter=meta_filter)
    return {
        'context'    : result['context'],
        'sources'    : [{'rank': c['rank'], 'agency': c['metadata'].get('agency',''),
                         'year': c['metadata'].get('year',''),
                         'project': c['metadata'].get('project_name',''),
                         'score': c['boosted_score']} for c in result['top_chunks']],
        'filter'     : result['meta_filter'],
        'sub_queries': result['sub_queries'],
    }

print('✅ get_context() 함수 준비 완료')

✅ get_context() 함수 준비 완료


In [14]:
import pandas as pd
from pathlib import Path

eval_files = sorted(Path('/home/euijeong/2Team_Project/hej/eval').glob('*.csv'))
df = pd.read_csv(eval_files[0])
print(df.columns.tolist())
print(df.head(2))

['id', 'type', 'difficulty', 'question', 'ground_truth_answer', 'ground_truth_docs', 'metadata_filter', 'history']
     id type difficulty                                           question  \
0  Q001    A          하     한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 얼마입니까?   
1  Q002    A          하  고려대학교에서 발주한 '차세대 포털·학사 정보시스템 구축사업'의 예산은 얼마로 배정...   

                 ground_truth_answer                        ground_truth_docs  \
0  해당 사업의 예산 규모는 14,107,009,000원입니다.  ["한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp"]   
1     해당 사업의 예산은 11,270,000,000원입니다.       ["고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf"]   

        metadata_filter history  
0  {"agency": "한국가스공사"}      []  
1   {"agency": "고려대학교"}      []  


In [15]:
import ast
from pathlib import Path

# 첫 번째 질문 직접 검색해보기
result = retriever.retrieve('한국가스공사의 차세대 통합정보시스템(ERP) 구축 사업 예산 규모는 얼마입니까?', verbose=True)

print('\n[Retrieved source_file]')
for c in result['top_chunks']:
    print(c['metadata'].get('source_file', '없음'))

print('\n[GT]')
gt_raw = '["한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp"]'
gt_docs = ast.literal_eval(gt_raw)
for g in gt_docs:
    print(Path(g).stem)

[①] 메타 필터 : {'agency': '한국가스공사'}
[①-1] 단일 쿼리


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.27it/s]


[②] Dense 6개 / [③] Sparse 10개


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.95it/s]


[⑦] Top-5 선정 완료

[Retrieved source_file]
한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp

[GT]
한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축


In [16]:
result2 = retriever.retrieve(
    '한국가스공사의 차세대 통합정보시스템(ERP) 구축 사업 예산 규모는 얼마입니까?',
    meta_filter={},  # 필터 없이
    verbose=True
)
print('\n[Retrieved source_file]')
for c in result2['top_chunks']:
    print(c['metadata'].get('source_file', '없음'))

[①] 메타 필터 : {}
[①-1] 단일 쿼리


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.91it/s]


[②] Dense 15개 / [③] Sparse 15개


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.69it/s]


[⑦] Top-5 선정 완료

[Retrieved source_file]
한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
한국기계연구원_(재공고)기계(연) 차세대 통합정보시스템 구축.hwp
한국기계연구원_기계(연) 차세대 통합정보시스템 구축.hwp
한국기계연구원_수의)(재공고)기계(연) 차세대 통합정보시스템 구축.hwp
한국의료분쟁조정중재원_차세대 통합정보시스템 구축을 위한 ISP 사업.hwp


In [17]:
# 한국가스공사 청크 몇 개나 있는지
result = collection.get(where={'agency': {'$eq': '한국가스공사'}})
print(f'한국가스공사 청크 수: {len(result["ids"])}')

# 메타데이터 샘플
if result['metadatas']:
    print(result['metadatas'][0])

한국가스공사 청크 수: 6
{'domains': ['통합정보시스템', 'ERP'], 'has_table': False, 'agency': '한국가스공사', 'organization_raw': '한국가스공사', 'organization_cleaned': '한국가스공사', 'has_number': False, 'year': '2024', 'source_file': '한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', 'original_name': '한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', 'project_name': '[재공고]차세대 통합정보시스템(ERP) 구축'}


In [18]:
gas_chunks = [c for c in ALL_CHUNKS if c['metadata'].get('agency') == '한국가스공사']
print(f'ALL_CHUNKS 내 한국가스공사: {len(gas_chunks)}개')

# 전체 기관 수
agencies = set(c['metadata'].get('agency') for c in ALL_CHUNKS)
print(f'전체 기관 수: {len(agencies)}개')
print(f'전체 청크 수: {len(ALL_CHUNKS)}개')

ALL_CHUNKS 내 한국가스공사: 6개
전체 기관 수: 404개
전체 청크 수: 16248개


In [19]:
# ======================================================================
# [CELL 11] FAIL 케이스 분석
# ======================================================================
fail_df = result_df[result_df['hit'] == False].copy()

print(f'❌ FAIL 케이스 수 : {len(fail_df)}개')
print('\n[타입별 FAIL 수]')
print(fail_df['type'].value_counts().to_string())
print('\n[난이도별 FAIL 수]')
print(fail_df['difficulty'].value_counts().to_string())

# 쿼리 분해 발생 여부
result_df['decomposed'] = result_df['sub_queries'].apply(
    lambda x: len(x) > 1 if isinstance(x, list) else False
)
fail_df = result_df[result_df['hit'] == False].copy()

# B타입 FAIL 중 쿼리 분해 발생 여부
b_fail = fail_df[fail_df['type'] == 'B']
print(f'\n[B타입 FAIL {len(b_fail)}건 중 쿼리 분해 발생 여부]')
print(b_fail['decomposed'].value_counts().to_string())

print(f'\n── B타입 FAIL 샘플 (Top-5) ──')
for _, row in b_fail.head(5).iterrows():
    print(f'  ID: {row["id"]}')
    print(f'    GT       : {row["gt_docs"]}')
    print(f'    retrieved: {row["retrieved"]}')
    print(f'    분해     : {row["sub_queries"]}')
    print()

❌ FAIL 케이스 수 : 191개

[타입별 FAIL 수]
type
E    48
B    42
C    40
D    33
A    28

[난이도별 FAIL 수]
difficulty
상     83
중     72
하     29
최상     7

[B타입 FAIL 42건 중 쿼리 분해 발생 여부]
decomposed
False    28
True     14

── B타입 FAIL 샘플 (Top-5) ──
  ID: Q034
    GT       : ['KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원', '사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스', '한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업']
    retrieved: ['KOICA 전자조달_[지문] [국제] (재공고)우즈베키스탄 ICT기반의 수자원정보']
    분해     : ["코이카의 '우즈베키스탄 방송시스템 사업', 아시아물위원회의 '우즈벡-키르기즈스탄 관개시스템 사업', 수출입은행의 '모잠비크 ITS 타당성 조사 사업'은 모두 중앙 및 해외 기관 원조 형식으로 수행되는 사업입니다. 이 세 사업의 주 대상국과 핵심 인프라 분야를 정리하고, 해당 예산 규모를 큰 금액부터 내림차순으로 나열하십시오."]

  ID: Q052
    GT       : ['사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스', '한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업']
    retrieved: ['한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성']
    분해     : ['한국수출입은행의 지능형교통시스템 사업타당성조사와 아시아물위원회의 스마트 관개시스템 구축사업이 공통적으로 목표 지원하는 수행 대상 국가 지역을 대조해 정리해 주십시오.']

  ID: Q053
    GT       : ['고려대학교_차세대 포털·학사 정보시스템 구축사업', 'KOICA 전자조달

In [20]:
# ======================================================================
# [CELL 11-C] C타입 히스토리 적용 전용 평가
# ======================================================================
# C타입(후속 질문)은 히스토리 없이 평가하면 검색기 실패처럼 보이지만
# 실제로는 컨텍스트 부재가 원인.
# history를 쿼리에 합성해서 실제 프로덕트 환경 기준 성능을 측정.
# 로컬 기준: 히스토리 적용 후 Hit@5 +15.33%p 향상 확인 (v7 평가).
# ======================================================================
import json as _json

def parse_history(raw) -> list:
    """history 컬럼 파싱. role/content 또는 user/assistant 형식 모두 처리."""
    if not raw or (isinstance(raw, float)):
        return []
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw or raw in ['[]', 'null']:
            return []
        try:
            parsed = _json.loads(raw)
        except:
            return []
    else:
        parsed = raw

    result = []
    for h in parsed:
        if not isinstance(h, dict):
            continue
        # role/content 형식
        if 'role' in h and 'content' in h:
            result.append({'role': h['role'], 'content': h['content']})
        # user/assistant 형식
        elif 'user' in h:
            result.append({'role': 'user', 'content': h['user']})
        elif 'assistant' in h:
            result.append({'role': 'assistant', 'content': h['assistant']})
    return result


# C타입만 필터
c_df = eval_df[eval_df['type'] == 'C'].copy()

# 히스토리 없는 케이스 제외
c_df['parsed_history'] = c_df['history'].apply(parse_history)
c_with_history = c_df[c_df['parsed_history'].apply(lambda h: len(h) > 0)].copy()

print(f'✅ C타입 전체 : {len(c_df)}개')
print(f'✅ 히스토리 있음 : {len(c_with_history)}개 (없는 {len(c_df)-len(c_with_history)}개 제외)')

results_c = []
for _, row in tqdm(c_with_history.iterrows(), total=len(c_with_history), desc='C타입 히스토리 평가'):
    query   = row['question']
    history = row['parsed_history']

    gt_raw = row.get('ground_truth_docs', '')
    try:    gt_docs = ast.literal_eval(gt_raw) if isinstance(gt_raw, str) else []
    except: gt_docs = [gt_raw] if isinstance(gt_raw, str) and gt_raw else []

    mf_raw = row.get('metadata_filter', {})
    try:    meta_filter = ast.literal_eval(mf_raw) if isinstance(mf_raw, str) else {}
    except: meta_filter = {}

    # 히스토리 마지막 user 발화를 현재 쿼리 앞에 합성
    prev_user = [h['content'] for h in history if h['role'] == 'user']
    effective_query = f'{prev_user[-1]} {query}' if prev_user else query

    result = retriever.retrieve(effective_query, meta_filter=meta_filter or None)
    retrieved_stems = [normalize_filename(c['metadata'].get('source_file', '')) for c in result['top_chunks']]
    gt_stems        = [normalize_filename(g) for g in gt_docs]

    results_c.append({
        'id'        : row['id'],
        'difficulty': row['difficulty'],
        'hit'       : calc_hit(gt_stems, retrieved_stems),
        'mrr'       : calc_mrr(gt_stems, retrieved_stems),
        'ndcg'      : calc_ndcg(gt_stems, retrieved_stems),
        'retrieved' : retrieved_stems,
        'gt_docs'   : gt_stems,
    })

c_result_df = pd.DataFrame(results_c)
c_subset    = c_result_df[c_result_df['hit'].notna()]

c_hit = c_subset['hit'].mean()
c_mrr = c_subset['mrr'].mean()
c_ndcg= c_subset['ndcg'].mean()

# 히스토리 미적용 C타입 수치 (CELL 9 결과에서)
c_before_hit  = eval_subset[eval_subset['type']=='C']['hit'].mean()
c_before_mrr  = eval_subset[eval_subset['type']=='C']['mrr'].mean()
c_before_ndcg = eval_subset[eval_subset['type']=='C']['ndcg'].mean()

print('\n' + '='*55)
print(f'📊 C타입 히스토리 적용  Hit@5 : {c_hit:.4f}  ({c_hit*100:.2f}%)')
print(f'📊 C타입 히스토리 적용  MRR   : {c_mrr:.4f}')
print(f'📊 C타입 히스토리 적용  nDCG  : {c_ndcg:.4f}')
print('='*55)

print('\n[히스토리 적용 전후 Delta]')
print(f'{"지표":>6}  {"적용 전":>8}  {"적용 후":>8}  {"Delta":>8}')
print(f'{"Hit@5":>6}  {c_before_hit:>8.4f}  {c_hit:>8.4f}  {c_hit-c_before_hit:>+8.4f}')
print(f'{"MRR":>6}  {c_before_mrr:>8.4f}  {c_mrr:>8.4f}  {c_mrr-c_before_mrr:>+8.4f}')
print(f'{"nDCG":>6}  {c_before_ndcg:>8.4f}  {c_ndcg:>8.4f}  {c_ndcg-c_before_ndcg:>+8.4f}')

print('\n[난이도별]')
print(f'{"난이도":>4}  {"Hit@5":>7} {"MRR":>7} {"nDCG":>7}  n')
for d, g in c_subset.groupby('difficulty'):
    print(f'  {d:>2}  {g["hit"].mean():>7.4f} {g["mrr"].mean():>7.4f} {g["ndcg"].mean():>7.4f}  {len(g)}')

# 저장
c_save_path = RESULT_DIR / f'eval_results_C타입_history_v{next_v}.csv'
c_result_df.to_csv(c_save_path, index=False, encoding='utf-8-sig')
print(f'\n✅ C타입 히스토리 평가 저장 : {c_save_path}')
print(f'   ({len(c_with_history)}개 평가 / history 없는 C타입 {len(c_df)-len(c_with_history)}개 제외)')

✅ C타입 전체 : 143개
✅ 히스토리 있음 : 129개 (없는 14개 제외)


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.03it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.79it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.80it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 48.43it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.23it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 55.06it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.84it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 52.67it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.67it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 54.15it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.40it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 55.50it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 47.71it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.62it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.21it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.24it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 54.69it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.1


📊 C타입 히스토리 적용  Hit@5 : 0.8217  (82.17%)
📊 C타입 히스토리 적용  MRR   : 0.7313
📊 C타입 히스토리 적용  nDCG  : 0.7549

[히스토리 적용 전후 Delta]
    지표      적용 전      적용 후     Delta
 Hit@5    0.7203    0.8217   +0.1014
   MRR    0.6311    0.7313   +0.1001
  nDCG    0.6544    0.7549   +0.1005

[난이도별]
 난이도    Hit@5     MRR    nDCG  n
   상   0.6957  0.6715  0.6777  69
   중   1.0000  0.8276  0.8727  29
   하   0.9355  0.7742  0.8164  31

✅ C타입 히스토리 평가 저장 : /home/euijeong/2Team_Project/hej/eval_results_C타입_history_v1.csv
   (129개 평가 / history 없는 C타입 14개 제외)


In [25]:
# ======================================================================
# [CELL 12] chunks_all.json 기준 평가 — kh_fixed_1200_200_v2 결과와 비교
# ======================================================================
import json

# ── chunks_all.json 로드 ──────────────────────────────────────
CHUNKS_ALL_PATH = PROJECT_ROOT / 'chunks' / 'chunks_all.json'
if not CHUNKS_ALL_PATH.exists():
    raise FileNotFoundError(f'chunks_all.json 없음: {CHUNKS_ALL_PATH}')

with open(CHUNKS_ALL_PATH, 'r', encoding='utf-8') as f:
    chunks_all_raw = json.load(f)
print(f'✅ chunks_all.json 로드 : {len(chunks_all_raw):,}개')

# ── chunk_id 중복 제거 ────────────────────────────────────────
seen_ids = set()
chunks_all = []
for c in chunks_all_raw:
    if c['chunk_id'] not in seen_ids:
        seen_ids.add(c['chunk_id'])
        chunks_all.append(c)

n_dup = len(chunks_all_raw) - len(chunks_all)
if n_dup > 0:
    print(f'⚠️  중복 {n_dup}개 제거 : {len(chunks_all_raw):,} → {len(chunks_all):,}개')

# ── ChromaDB 새 컬렉션으로 인덱싱 ────────────────────────────
COLLECTION_NAME_ALL = 'bidmate_chunks_all'
collection_all = chroma_client.get_or_create_collection(
    name     = COLLECTION_NAME_ALL,
    metadata = {'hnsw:space': 'cosine'},
)

if collection_all.count() == 0:
    print(f'🔄 KURE-v1 임베딩 + 인덱싱 시작 | 총 {len(chunks_all):,}개')
    for i in range(0, len(chunks_all), BATCH_SIZE):
        batch      = chunks_all[i : i + BATCH_SIZE]
        texts      = [c['text'] for c in batch]
        embeddings = embed_model.encode(texts, normalize_embeddings=True).tolist()
        collection_all.upsert(
            ids        = [c['chunk_id'] for c in batch],
            documents  = texts,
            metadatas  = [c['metadata'] for c in batch],
            embeddings = embeddings,
        )
        if (i // BATCH_SIZE) % 10 == 0:
            print(f'   진행 : {min(i+BATCH_SIZE, len(chunks_all)):,} / {len(chunks_all):,}')
    print(f'✅ 인덱싱 완료 | 문서 수 : {collection_all.count():,}')
else:
    print(f'✅ 기존 컬렉션 재사용 | 문서 수 : {collection_all.count():,}')

# ── BM25 재구축 ──────────────────────────────────────────────
texts_all    = [c['text']     for c in chunks_all]
bm25_ids_all = [c['chunk_id'] for c in chunks_all]
print(f'🔄 BM25 토크나이징 시작 | {len(texts_all):,}개')
tokenized_all = [tokenize_ko(t) for t in texts_all]
bm25_all      = BM25Okapi(tokenized_all)
print('✅ BM25 구축 완료')

# ── chunks_all 기준 retriever 생성 ───────────────────────────
agencies_all = list({c['metadata'].get('agency','') for c in chunks_all if c['metadata'].get('agency','')})

retriever_all = BidMateRetriever(
    collection     = collection_all,
    bm25_index     = bm25_all,
    bm25_chunk_ids = bm25_ids_all,
    bm25_texts     = texts_all,
    embed_model    = embed_model,
    all_chunks     = chunks_all,
    reranker       = reranker,
)
print('✅ chunks_all 기준 retriever 생성 완료')

# ── 평가 루프 ─────────────────────────────────────────────────
# ALL_AGENCIES를 chunks_all 기준으로 임시 교체
_orig_agencies = ALL_AGENCIES.copy()
ALL_AGENCIES.clear()
ALL_AGENCIES.extend(agencies_all)

results_all = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc='chunks_all 평가'):
    query  = row['question']
    gt_raw = row.get('ground_truth_docs', '')
    try:    gt_docs = ast.literal_eval(gt_raw) if isinstance(gt_raw, str) else []
    except: gt_docs = [gt_raw] if isinstance(gt_raw, str) and gt_raw else []
    mf_raw = row.get('metadata_filter', {})
    try:    meta_filter = ast.literal_eval(mf_raw) if isinstance(mf_raw, str) else {}
    except: meta_filter = {}

    result = retriever_all.retrieve(query, meta_filter=meta_filter or None)
    retrieved_stems = [normalize_filename(c['metadata'].get('source_file','')) for c in result['top_chunks']]
    gt_stems        = [normalize_filename(g) for g in gt_docs]

    results_all.append({
        'id'         : row['id'],
        'type'       : row['type'],
        'difficulty' : row['difficulty'],
        'hit'        : calc_hit(gt_stems, retrieved_stems),
        'mrr'        : calc_mrr(gt_stems, retrieved_stems),
        'ndcg'       : calc_ndcg(gt_stems, retrieved_stems),
        'retrieved'  : retrieved_stems,
        'gt_docs'    : gt_stems,
        'sub_queries': result['sub_queries'],
    })

# ALL_AGENCIES 원복
ALL_AGENCIES.clear()
ALL_AGENCIES.extend(_orig_agencies)

df_all  = pd.DataFrame(results_all)
sub_all = df_all[df_all['hit'].notna()]
h_all   = sub_all['hit'].mean()
m_all   = sub_all['mrr'].mean()
n_all   = sub_all['ndcg'].mean()

# CELL 9 결과
h_kh = oh
m_kh = om
n_kh = on_

print('\n' + '='*60)
print('📊 데이터 비교 — kh_fixed_1200_200_v2 vs chunks_all')
print('='*60)
print(f'{"지표":>6}  {"kh_v2":>8}  {"chunks_all":>10}  {"Delta":>8}')
print(f'{"Hit@5":>6}  {h_kh:>8.4f}  {h_all:>10.4f}  {h_all-h_kh:>+8.4f}')
print(f'{"MRR":>6}  {m_kh:>8.4f}  {m_all:>10.4f}  {m_all-m_kh:>+8.4f}')
print(f'{"nDCG":>6}  {n_kh:>8.4f}  {n_all:>10.4f}  {n_all-n_kh:>+8.4f}')

print('\n[타입별]')
print(f'{"타입":>4}  {"kh_v2 Hit":>9} {"all Hit":>9}  {"Delta":>7}')
for t in ['A','B','C','D','E']:
    kh_t  = eval_subset[eval_subset['type']==t]['hit'].mean()
    all_t = sub_all[sub_all['type']==t]['hit'].mean()
    print(f'  {t:>2}  {kh_t:>9.4f} {all_t:>9.4f}  {all_t-kh_t:>+7.4f}')

# 저장
RESULT_SAVE_ALL = RESULT_DIR / f'eval_results_chunks_all_v{next_v}.csv'
df_all.to_csv(RESULT_SAVE_ALL, index=False, encoding='utf-8-sig')
print(f'\n✅ 저장 : {RESULT_SAVE_ALL}')


✅ chunks_all.json 로드 : 10,127개
⚠️  중복 59개 제거 : 10,127 → 10,068개
🔄 KURE-v1 임베딩 + 인덱싱 시작 | 총 10,068개


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.83it/s]


   진행 : 64 / 10,068


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.61it/s]


   진행 : 704 / 10,068


...

   진행 : 9,664 / 10,068


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.79it/s]


✅ 인덱싱 완료 | 문서 수 : 10,068
🔄 BM25 토크나이징 시작 | 10,068개
✅ BM25 구축 완료
✅ chunks_all 기준 retriever 생성 완료


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.95it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.33it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.17it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.42it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.34it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.54it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.76it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.61it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.63it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 58.57it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.74it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 55.81it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.40it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 58.06it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.42it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.16it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.02it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.2


📊 데이터 비교 — kh_fixed_1200_200_v2 vs chunks_all
    지표     kh_v2  chunks_all     Delta
 Hit@5    0.8264      0.8691   +0.0427
   MRR    0.7375      0.8020   +0.0645
  nDCG    0.7043      0.7729   +0.0686

[타입별]
  타입  kh_v2 Hit   all Hit    Delta
   A     0.9146    0.9207  +0.0061
   B     0.8650    0.8875  +0.0225
   C     0.7203    0.6993  -0.0210
   D     0.7963    0.8704  +0.0741
   E     0.6923    0.8782  +0.1859

✅ 저장 : /home/euijeong/2Team_Project/hej/eval_results_chunks_all_v1.csv


In [26]:
# ======================================================================
# [CELL 12-C] chunks_all 기준 C타입 히스토리 적용 평가
# ======================================================================
c_df_all = eval_df[eval_df['type'] == 'C'].copy()
c_df_all['parsed_history'] = c_df_all['history'].apply(parse_history)
c_with_history_all = c_df_all[c_df_all['parsed_history'].apply(lambda h: len(h) > 0)].copy()

print(f'✅ C타입 전체 : {len(c_df_all)}개')
print(f'✅ 히스토리 있음 : {len(c_with_history_all)}개')

_orig_agencies = ALL_AGENCIES.copy()
ALL_AGENCIES.clear()
ALL_AGENCIES.extend(agencies_all)

results_c_all = []
for _, row in tqdm(c_with_history_all.iterrows(), total=len(c_with_history_all), desc='chunks_all C타입 히스토리'):
    query   = row['question']
    history = row['parsed_history']
    gt_raw  = row.get('ground_truth_docs', '')
    try:    gt_docs = ast.literal_eval(gt_raw) if isinstance(gt_raw, str) else []
    except: gt_docs = [gt_raw] if isinstance(gt_raw, str) and gt_raw else []
    mf_raw = row.get('metadata_filter', {})
    try:    meta_filter = ast.literal_eval(mf_raw) if isinstance(mf_raw, str) else {}
    except: meta_filter = {}

    prev_user = [h['content'] for h in history if h['role'] == 'user']
    effective_query = f'{prev_user[-1]} {query}' if prev_user else query

    result = retriever_all.retrieve(effective_query, meta_filter=meta_filter or None)
    retrieved_stems = [normalize_filename(c['metadata'].get('source_file','')) for c in result['top_chunks']]
    gt_stems        = [normalize_filename(g) for g in gt_docs]

    results_c_all.append({
        'id'        : row['id'],
        'difficulty': row['difficulty'],
        'hit'       : calc_hit(gt_stems, retrieved_stems),
        'mrr'       : calc_mrr(gt_stems, retrieved_stems),
        'ndcg'      : calc_ndcg(gt_stems, retrieved_stems),
    })

ALL_AGENCIES.clear()
ALL_AGENCIES.extend(_orig_agencies)

c_all_df     = pd.DataFrame(results_c_all)
c_all_subset = c_all_df[c_all_df['hit'].notna()]
ch_all = c_all_subset['hit'].mean()
cm_all = c_all_subset['mrr'].mean()
cn_all = c_all_subset['ndcg'].mean()

# kh_v2 C타입 히스토리 결과
ch_kh = c_hit
cm_kh = c_mrr
cn_kh = c_ndcg

print('\n' + '='*60)
print('📊 C타입 히스토리 비교 — kh_v2 vs chunks_all')
print('='*60)
print(f'{"지표":>6}  {"kh_v2":>8}  {"chunks_all":>10}  {"Delta":>8}')
print(f'{"Hit@5":>6}  {ch_kh:>8.4f}  {ch_all:>10.4f}  {ch_all-ch_kh:>+8.4f}')
print(f'{"MRR":>6}  {cm_kh:>8.4f}  {cm_all:>10.4f}  {cm_all-cm_kh:>+8.4f}')
print(f'{"nDCG":>6}  {cn_kh:>8.4f}  {cn_all:>10.4f}  {cn_all-cn_kh:>+8.4f}')

✅ C타입 전체 : 143개
✅ 히스토리 있음 : 129개


Batches: 100%|██████████| 1/1 [00:00<00:00, 54.63it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 55.65it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.05it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 55.88it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.05it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 47.39it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.64it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.12it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.72it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.26it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.15it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.76it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 47.02it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.50it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 58.12it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.86it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.64it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 57.1


📊 C타입 히스토리 비교 — kh_v2 vs chunks_all
    지표     kh_v2  chunks_all     Delta
 Hit@5    0.8217      0.8605   +0.0388
   MRR    0.7313      0.7544   +0.0231
  nDCG    0.7549      0.7808   +0.0260
